# LeetCode 971 — Flip Binary Tree To Match Preorder Traversal bằng Python

Link đề: https://leetcode.com/problems/flip-binary-tree-to-match-preorder-traversal/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        ...
```

Ghi chú về đáp án: đề cho phép trả về danh sách các nút bị lật **theo thứ tự bất kỳ**; ta sẽ chứng minh (mục 10) rằng **tập** các nút cần lật là **duy nhất** (phép lật ở nút có ≤ 1 con không đổi duyệt tiền thứ tự, còn ở nút có 2 con thì thứ tự được `voyage` quyết định). Do đó mọi test so sánh sau khi `sorted(...)`, kể cả trường hợp không thể: `[-1]`. Notebook tự dựng `TreeNode`, `build_tree` kiểu LeetCode (danh sách theo tầng, `None` là nút trống) và `to_level`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lật ít nút nhất (đổi chỗ hai cây con) để duyệt tiền thứ tự (preorder) của cây trùng `voyage`; không thể thì trả `[-1]`.
2. Nắm công cụ: DFS tiền thứ tự với **con trỏ** vào `voyage`, ngăn xếp tường minh, ánh xạ vị trí `pos[val]`, chia đoạn `voyage` theo kích thước cây con.
3. Đi từ **thử mọi tập nút để lật** → **DFS có con trỏ** → **DFS lặp bằng stack** → **suy ra lật từ vị trí trong `voyage`** → **chia đoạn theo kích thước cây con**.
4. Hiểu mấu chốt: tại mỗi nút, đọc giá trị kế tiếp trong `voyage`: nếu nó **không** là con trái thì (nếu có con phải trùng) **bắt buộc lật**; mọi quyết định bị ép, nên **tham lam là tối ưu và duy nhất**.
5. Tránh lỗi kinh điển: lật nhầm nút chỉ có một con, trả danh sách lật dở dang thay vì `[-1]`, quên kiểm tra giá trị nút so với `voyage[i]`, ghi nhầm giá trị con thay vì giá trị nút cha, dùng con trỏ cục bộ (không dùng chung), đi đúng thứ tự trái–phải sau khi "quyết định lật".

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân có `n` nút, mỗi nút có giá trị **duy nhất** từ `1` đến `n`, và dãy `voyage` gồm `n` giá trị — duyệt tiền thứ tự **mong muốn**.

Có thể **lật** một nút bất kỳ bằng cách đổi chỗ cây con trái và phải của nó. Hãy lật **ít nút nhất** để duyệt tiền thứ tự của cây **khớp** `voyage`.

Trả về danh sách giá trị các nút bị lật (thứ tự bất kỳ). Nếu không thể, trả về `[-1]`.

**Ví dụ 1:** `root = [1,2]`, `voyage = [2,1]` → `[-1]` (gốc là `1` nhưng `voyage` bắt đầu bằng `2`).

**Ví dụ 2:** `root = [1,2,3]`, `voyage = [1,3,2]` → `[1]`.

**Ví dụ 3:** `root = [1,2,3]`, `voyage = [1,2,3]` → `[]`.

**Ràng buộc:** `n == voyage.length`; `1 <= n <= 100`; `1 <= Node.val, voyage[i] <= n`; giá trị trong cây duy nhất; giá trị trong `voyage` duy nhất.

## 3. Kiến thức nền

### 3.1 `TreeNode`, dựng cây kiểu LeetCode, duyệt tiền thứ tự

Tiền thứ tự: **nút, rồi cây con trái, rồi cây con phải**. Danh sách kiểu LeetCode được đọc theo tầng bằng hàng đợi (nút trống không có con trong danh sách). Ta tự dựng `TreeNode`, `build_tree`, `to_level` và `preorder`.

In [1]:
import itertools
import random
import time
from collections import deque
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level(root):
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def preorder(root):
    out = []

    def go(n):
        if n is not None:
            out.append(n.val)
            go(n.left)
            go(n.right)

    go(root)
    return out


def tup(root):
    return None if root is None else (root.val, tup(root.left), tup(root.right))


def from_tuple(t):
    return None if t is None else TreeNode(t[0], from_tuple(t[1]), from_tuple(t[2]))


def size(root):
    return 0 if root is None else 1 + size(root.left) + size(root.right)


r = build_tree([1, 2, 3, 4, 5, 6, 7])
print(preorder(r), to_level(r), tup(build_tree([1, None, 2])))
assert to_level(build_tree([1, 2, 3, None, 4])) == [1, 2, 3, None, 4]

[1, 2, 4, 5, 3, 6, 7] [1, 2, 3, 4, 5, 6, 7] (1, None, (2, None, None))


### 3.2 Áp dụng một tập phép lật và kiểm tra

`apply_flips(root, flips)` trả về **bản sao** của cây trong đó mọi nút có giá trị thuộc `flips` bị đổi chỗ hai con. Đây là cách **kiểm tra tính hợp lệ** của một đáp án: sau khi lật, `preorder` phải bằng `voyage`.

In [2]:
def apply_flips(root, flips):
    flips = set(flips)

    def go(n):
        if n is None:
            return None
        l, r = go(n.left), go(n.right)
        if n.val in flips:
            l, r = r, l
        return TreeNode(n.val, l, r)

    return go(root)


root = build_tree([1, 2, 3])
print(preorder(root), "-> lật 1 ->", preorder(apply_flips(root, [1])))
print("gốc không bị sửa:", preorder(root))

[1, 2, 3] -> lật 1 -> [1, 3, 2]
gốc không bị sửa: [1, 2, 3]


### 3.3 Lật ở nút chỉ có một con (hoặc lá) không đổi preorder

Nếu nút có **một** con, lật đổi con đó sang bên kia nhưng tiền thứ tự vẫn là "nút, rồi con". Lá: không có gì để đổi. Chỉ nút có **hai** con mới làm thay đổi thứ tự.

In [3]:
only_right = build_tree([1, None, 2, 3])          # 1 có con phải 2; 2 có con trái 3
print(preorder(only_right), preorder(apply_flips(only_right, [1])), preorder(apply_flips(only_right, [2])), preorder(apply_flips(only_right, [3])))
assert preorder(only_right) == preorder(apply_flips(only_right, [1]))

[1, 2, 3] [1, 2, 3] [1, 2, 3] [1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

`root = [1,2,3]`, `voyage = [1,3,2]`. Đi tiền thứ tự cùng một con trỏ `i` trên `voyage`:

- Tại `1`: `voyage[0] = 1` khớp. Con trái là `2`, nhưng `voyage[1] = 3` ≠ `2` ⇒ để khớp phải **lật** `1`, tức đi vào con phải `3` trước. `3` khớp `voyage[1]`; rồi con (cũ) trái `2` khớp `voyage[2]`. Kết quả `[1]`.

Nếu `voyage[1]` không bằng cả `2` lẫn `3` thì không có cách nào ⇒ `[-1]`. Mỗi bước bị **ép**: giá trị kế tiếp trong `voyage` phải là gốc của cây con được duyệt tiếp; nó chỉ có thể là con trái hoặc con phải.

In [4]:
root, voyage = build_tree([1, 2, 3]), [1, 3, 2]
print("cây:", tup(root), "| voyage:", voyage)
print("voyage[1] =", voyage[1], "| con trái =", root.left.val, "| con phải =", root.right.val, "=> con phải đi trước => lật nút", root.val)

cây: (1, (2, None, None), (3, None, None)) | voyage: [1, 3, 2]
voyage[1] = 3 | con trái = 2 | con phải = 3 => con phải đi trước => lật nút 1


## 5. Brute force — thử mọi tập nút để lật

**Ý tưởng:** theo đúng định nghĩa: với `k = 0, 1, 2, …`, thử mọi tập `k` nút (theo giá trị) để lật; nếu `preorder(apply_flips(root, S)) == voyage` thì ghi nhận. Dừng ở `k` nhỏ nhất có nghiệm; trả `[-1]` nếu không `k` nào cho nghiệm. Hàm cũng **khẳng định tập nghiệm nhỏ nhất là duy nhất** (ném lỗi nếu có hai tập cùng cỡ) — một phát biểu ta sẽ chứng minh ở mục 10. Số tập `2^n` ⇒ chỉ cho `n ≲ 12`.

In [5]:
def node_values(root):
    out, stack = [], [root]
    while stack:
        n = stack.pop()
        if n:
            out.append(n.val)
            stack.extend([n.left, n.right])
    return sorted(out)


def flip_brute(root, voyage):
    vals = node_values(root)
    for k in range(len(vals) + 1):
        found = [c for c in itertools.combinations(vals, k) if preorder(apply_flips(root, c)) == list(voyage)]
        if found:
            assert len(found) == 1, f"có {len(found)} tập nghiệm cùng cỡ {k}: {found}"
            return list(found[0])
    return [-1]


print(flip_brute(build_tree([1, 2]), [2, 1]), flip_brute(build_tree([1, 2, 3]), [1, 3, 2]), flip_brute(build_tree([1, 2, 3]), [1, 2, 3]))

[-1] [1] []


In [6]:
# Code nộp LeetCode cho Brute force (chỉ để minh hoạ; mũ theo số nút)
import itertools


class SolutionBrute:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        def values(n, acc):
            if n:
                acc.append(n.val)
                values(n.left, acc)
                values(n.right, acc)
            return acc

        def pre(n, flips, acc):
            if n:
                acc.append(n.val)
                a, b = (n.right, n.left) if n.val in flips else (n.left, n.right)
                pre(a, flips, acc)
                pre(b, flips, acc)
            return acc

        vals = sorted(values(root, []))
        for k in range(len(vals) + 1):
            for c in itertools.combinations(vals, k):
                if pre(root, set(c), []) == voyage:
                    return list(c)
        return [-1]

**Độ phức tạp:** `O(2^n · n)` thời gian, `O(n)` bộ nhớ. Với `n = 100` bất khả thi.

**Nút thắt:** ta thử cả `2^n` tập trong khi tại mỗi nút `voyage` đã **quyết định** nên lật hay không.

## 6. Bước đột phá

Duyệt tiền thứ tự với con trỏ `i` vào `voyage`. Tại nút `u` (đã khớp `voyage[i]`, rồi `i += 1`): nếu `u` có con trái và `u.left.val != voyage[i]` thì **lật `u`** (ghi `u.val`, đi phải trước); ngược lại đi trái trước. Bất kỳ lúc nào `u.val != voyage[i]` ⇒ `[-1]`. Mỗi nút làm `O(1)` việc ⇒ `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DFS đệ quy có con trỏ (đáp án chính), **Cách 3** DFS lặp bằng stack, **Cách 4** suy ra lật từ vị trí trong `voyage`, **Cách 5** chia đoạn `voyage` theo kích thước cây con.

### Cách 2 — DFS đệ quy có con trỏ

**Giải thích từng đoạn:**

1. `idx` là con trỏ vào `voyage` (biến `nonlocal`, **dùng chung** cho mọi lời gọi).
2. `dfs(node)`: nút rỗng ⇒ `True`; `node.val != voyage[idx]` ⇒ `False`; tăng `idx`.
3. Nếu `node.left` tồn tại và `node.left.val != voyage[idx]`: ghi `node.val` vào `flips`, đi **phải rồi trái**; ngược lại **trái rồi phải**.
4. Kết quả: `flips` nếu `dfs(root)` thành công, ngược lại `[-1]` (bỏ danh sách lật dở dang).

In [7]:
def flip_dfs(root, voyage):
    flips = []
    idx = 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)                   # phải lật: giá trị kế tiếp không phải con trái
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    return flips if dfs(root) else [-1]


for (l, v), want in [(([1, 2], [2, 1]), [-1]), (([1, 2, 3], [1, 3, 2]), [1]), (([1, 2, 3], [1, 2, 3]), [])]:
    assert flip_dfs(build_tree(l), v) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDFS:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        flips, idx = [], 0

        def dfs(node):
            nonlocal idx
            if node is None:
                return True
            if idx >= len(voyage) or node.val != voyage[idx]:
                return False
            idx += 1
            if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
                flips.append(node.val)
                return dfs(node.right) and dfs(node.left)
            return dfs(node.left) and dfs(node.right)

        return flips if dfs(root) else [-1]

**Độ phức tạp:** `O(n)` thời gian, `O(h)` ngăn xếp đệ quy (`h ≤ n ≤ 100`).

### Cách 3 — DFS lặp bằng stack

**Ý tưởng:** đẩy `root`; lấy nút ra, so với `voyage[i]`; nếu cần lật thì đẩy **trái trước rồi phải** (phải nằm trên đỉnh nên được xử lý trước); không lật thì đẩy phải trước rồi trái.

In [9]:
def flip_iter(root, voyage):
    flips = []
    i = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if i >= len(voyage) or node.val != voyage[i]:
            return [-1]
        i += 1
        if node.left and i < len(voyage) and node.left.val != voyage[i]:
            flips.append(node.val)
            stack.append(node.left)                # trái đẩy trước ...
            stack.append(node.right)               # ... phải nằm trên đỉnh: xử lý trước
        else:
            stack.append(node.right)
            stack.append(node.left)
    return flips


for (l, v), want in [(([1, 2], [2, 1]), [-1]), (([1, 2, 3], [1, 3, 2]), [1]), (([1, 2, 3], [1, 2, 3]), [])]:
    assert flip_iter(build_tree(l), v) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionIter:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        flips, i, stack = [], 0, [root]
        while stack:
            node = stack.pop()
            if node is None:
                continue
            if i >= len(voyage) or node.val != voyage[i]:
                return [-1]
            i += 1
            if node.left and i < len(voyage) and node.left.val != voyage[i]:
                flips.append(node.val)
                stack.append(node.left)
                stack.append(node.right)
            else:
                stack.append(node.right)
                stack.append(node.left)
        return flips

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (stack).

### Cách 4 — Suy ra lật từ vị trí trong `voyage`

**Ý tưởng:** lập `pos[val]` = vị trí của `val` trong `voyage`. Nút có **hai** con cần lật ⇔ `pos[left] > pos[right]` (con phải xuất hiện trước). Duyệt cây theo đúng thứ tự "đã lật" để dựng dãy `order` rồi so với `voyage`: nếu khác ⇒ `[-1]`. (Bước so sánh cuối là bắt buộc: vị trí tương đối của hai anh em chưa đủ để chứng minh toàn dãy khớp.)

In [11]:
def flip_positions(root, voyage):
    pos = {v: i for i, v in enumerate(voyage)}
    flips, order = [], []
    stack = [root]
    while stack:
        node = stack.pop()
        if node.val not in pos:
            return [-1]
        order.append(node.val)
        kids = [c for c in (node.left, node.right) if c is not None]
        if len(kids) == 2 and pos.get(kids[0].val, -1) > pos.get(kids[1].val, -1):
            flips.append(node.val)
            kids.reverse()                               # thứ tự thăm mong muốn
        stack.extend(reversed(kids))                     # kids[0] nằm trên đỉnh stack
    return flips if order == list(voyage) else [-1]


for (l, v), want in [(([1, 2], [2, 1]), [-1]), (([1, 2, 3], [1, 3, 2]), [1]), (([1, 2, 3], [1, 2, 3]), [])]:
    assert flip_positions(build_tree(l), v) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionPositions:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        pos = {v: i for i, v in enumerate(voyage)}
        flips, order, stack = [], [], [root]
        while stack:
            node = stack.pop()
            if node.val not in pos:
                return [-1]
            order.append(node.val)
            kids = [c for c in (node.left, node.right) if c]
            if len(kids) == 2 and pos.get(kids[0].val, -1) > pos.get(kids[1].val, -1):
                flips.append(node.val)
                kids.reverse()
            stack.extend(reversed(kids))
        return flips if order == voyage else [-1]

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — Chia đoạn `voyage` theo kích thước cây con

**Ý tưởng:** `voyage[lo : lo + size(node)]` phải là tiền thứ tự của cây con tại `node`. Biết `size` của hai con: nếu `voyage[lo+1] == left.val` thì cây con trái chiếm `[lo+1, lo+1+sl)` và phải chiếm phần còn lại; ngược lại **lật**, cây con phải chiếm `[lo+1, lo+1+sr)`, trái phần còn lại. Mỗi lời gọi chỉ cần so **một** giá trị.

In [13]:
def flip_sizes(root, voyage):
    n = len(voyage)
    sz = {}

    def calc(node):
        if node is None:
            return 0
        sz[id(node)] = 1 + calc(node.left) + calc(node.right)
        return sz[id(node)]

    calc(root)
    flips = []

    def solve(node, lo):
        if node is None:
            return True
        if lo >= n or voyage[lo] != node.val:
            return False
        l, r = node.left, node.right
        if l and r:
            sl, sr = sz[id(l)], sz[id(r)]
            if lo + 1 < n and voyage[lo + 1] == l.val:
                return solve(l, lo + 1) and solve(r, lo + 1 + sl)
            flips.append(node.val)
            return solve(r, lo + 1) and solve(l, lo + 1 + sr)
        return solve(l or r, lo + 1)

    return flips if solve(root, 0) else [-1]


for (l, v), want in [(([1, 2], [2, 1]), [-1]), (([1, 2, 3], [1, 3, 2]), [1]), (([1, 2, 3], [1, 2, 3]), [])]:
    assert flip_sizes(build_tree(l), v) == want
print("3 ví dụ của đề: đúng")

3 ví dụ của đề: đúng


In [14]:
# Code nộp LeetCode cho Cách 5
class SolutionSizes:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        n, sz, flips = len(voyage), {}, []

        def calc(node):
            if node is None:
                return 0
            sz[id(node)] = 1 + calc(node.left) + calc(node.right)
            return sz[id(node)]

        def solve(node, lo):
            if node is None:
                return True
            if lo >= n or voyage[lo] != node.val:
                return False
            l, r = node.left, node.right
            if l and r:
                if lo + 1 < n and voyage[lo + 1] == l.val:
                    return solve(l, lo + 1) and solve(r, lo + 1 + sz[id(l)])
                flips.append(node.val)
                return solve(r, lo + 1) and solve(l, lo + 1 + sz[id(r)])
            return solve(l or r, lo + 1)

        calc(root)
        return flips if solve(root, 0) else [-1]

**Độ phức tạp:** `O(n)` thời gian (tính kích thước + duyệt), `O(n)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `root = [1,2,3,4,5,6,7]`, `voyage = [1,3,6,7,2,5,4]`:

| bước | nút | `voyage[idx]` | quyết định |
|---|---|---|---|
| 1 | `1` | `1` ✓ | con trái `2` ≠ `voyage[1] = 3` ⇒ **lật `1`**, đi phải |
| 2 | `3` | `3` ✓ | con trái `6` = `voyage[2]` ⇒ không lật |
| 3 | `6` | `6` ✓ | lá |
| 4 | `7` | `7` ✓ | lá |
| 5 | `2` | `2` ✓ | con trái `4` ≠ `voyage[5] = 5` ⇒ **lật `2`**, đi phải |
| 6 | `5` | `5` ✓ | lá |
| 7 | `4` | `4` ✓ | lá |

Kết quả `[1, 2]`. Hàm trace in từng bước:

In [15]:
def trace_dfs(root, voyage):
    print("voyage =", voyage)
    flips, idx = [], 0

    def dfs(node, depth):
        nonlocal idx
        pad = "  " * depth
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            print(f"{pad}nút {node.val}: voyage[{idx}] = {voyage[idx] if idx < len(voyage) else None} -> KHÔNG khớp => [-1]")
            return False
        print(f"{pad}nút {node.val}: khớp voyage[{idx}] = {voyage[idx]}")
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            print(f"{pad}  con trái {node.left.val} != voyage[{idx}] = {voyage[idx]} => LẬT {node.val}, đi phải trước")
            flips.append(node.val)
            return dfs(node.right, depth + 1) and dfs(node.left, depth + 1)
        return dfs(node.left, depth + 1) and dfs(node.right, depth + 1)

    ok = dfs(root, 0)
    res = flips if ok else [-1]
    print("=>", res)
    return res


assert trace_dfs(build_tree([1, 2, 3, 4, 5, 6, 7]), [1, 3, 6, 7, 2, 5, 4]) == [1, 2]

voyage = [1, 3, 6, 7, 2, 5, 4]
nút 1: khớp voyage[0] = 1
  con trái 2 != voyage[1] = 3 => LẬT 1, đi phải trước
  nút 3: khớp voyage[1] = 3
    nút 6: khớp voyage[2] = 6
    nút 7: khớp voyage[3] = 7
  nút 2: khớp voyage[4] = 2
    con trái 4 != voyage[5] = 5 => LẬT 2, đi phải trước
    nút 5: khớp voyage[5] = 5
    nút 4: khớp voyage[6] = 4
=> [1, 2]


**Trace trường hợp `[-1]` dù đã lật dở dang:** `root = [1,2,3]`, `voyage = [1,3,4]`: sau khi lật `1` (vì `2 ≠ 3`), nút `3` khớp, rồi nút `2` so với `voyage[2] = 4` ✗ ⇒ phải trả `[-1]`, **không** trả `[1]`.

In [16]:
assert trace_dfs(build_tree([1, 2, 3]), [1, 3, 4]) == [-1]

voyage = [1, 3, 4]
nút 1: khớp voyage[0] = 1
  con trái 2 != voyage[1] = 3 => LẬT 1, đi phải trước
  nút 3: khớp voyage[1] = 3
  nút 2: voyage[2] = 4 -> KHÔNG khớp => [-1]
=> [-1]


## 9. Code nộp LeetCode

Chọn **Cách 2** (DFS có con trỏ): ngắn, đúng, dễ nhớ. Giữ nguyên `class Solution`, `flipMatchVoyage(self, root, voyage)`:

In [17]:
class Solution:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        flips, idx = [], 0                                    # idx: con trỏ dùng chung vào voyage

        def dfs(node):
            nonlocal idx
            if node is None:
                return True
            if idx >= len(voyage) or node.val != voyage[idx]:  # giá trị nút phải bằng giá trị mong muốn
                return False
            idx += 1
            if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
                flips.append(node.val)                         # phải lật nút này
                return dfs(node.right) and dfs(node.left)
            return dfs(node.left) and dfs(node.right)

        return flips if dfs(root) else [-1]                    # thất bại: bỏ danh sách dở dang


ex = [(([1, 2], [2, 1]), [-1]), (([1, 2, 3], [1, 3, 2]), [1]), (([1, 2, 3], [1, 2, 3]), [])]
print([Solution().flipMatchVoyage(build_tree(l), v) for (l, v), _ in ex])
assert all(Solution().flipMatchVoyage(build_tree(l), v) == w for (l, v), w in ex)

[[-1], [1], []]


## 10. Vì sao đúng

**(a) Tập lật tối thiểu là duy nhất và chỉ gồm nút có hai con.** Lật nút `u` đổi chỗ hai cây con của `u`. Nếu `u` có ≤ 1 con thì tiền thứ tự không đổi (mục 3.3), nên bỏ `u` khỏi tập lật vẫn khớp ⇒ `u` không thuộc tập tối thiểu. Nếu `u` có hai con `L`, `R`: tiền thứ tự của cây con tại `u` là `u`, rồi preorder(con đi trước), rồi preorder(con đi sau). Trong `voyage`, nút ngay sau `u` (trong đoạn của cây con `u`) phải là gốc của con đi trước; gốc của `L` và `R` là hai giá trị **khác nhau**, nên chỉ **một** trong hai thứ tự khớp ⇒ việc lật `u` hay không bị **ép**. Do đó với mỗi `voyage` có tối đa một tập lật làm preorder khớp mà không chứa nút ≤ 1 con, và tập tối thiểu chính là tập đó.

**(b) Thuật toán tham lam đúng.** Con trỏ `idx` luôn là vị trí trong `voyage` của nút đầu tiên chưa thăm theo thứ tự đã chọn. Tại nút `u` có hai con, `voyage[idx]` (sau khi tăng) phải là gốc của một trong hai con; nó bằng `u.left.val` ⇒ đi trái trước; khác ⇒ đi phải trước (nếu cũng không khớp phải thì bước kế tiếp phát hiện `val != voyage[idx]` và trả `False`). Nếu `u` chỉ có con trái, điều kiện `node.left.val != voyage[idx]` cho kết quả sai (lật) **chỉ khi** giá trị không khớp, mà khi đó nhánh sau cũng thất bại ⇒ `[-1]`; nếu chỉ có con phải thì `node.left` là `None` nên không lật. Vậy **không bao giờ lật nhầm nút một con**.

**(c) `[-1]` đúng.** Nếu tồn tại cách lật để khớp thì theo (a) nó là chính tập ép buộc mà DFS dựng; DFS thất bại ⇒ không tồn tại ⇒ `[-1]`. Ta cũng phải bỏ danh sách lật dở dang khi thất bại (đề yêu cầu trả đúng `[-1]`).

**(d) `n == len(voyage)`.** Số nút bằng độ dài `voyage` nên khi còn nút chưa thăm luôn còn `voyage[idx]`; ta vẫn bảo vệ `idx < len(voyage)` để an toàn.

**(e) Các Cách 3, 4, 5.** Cách 3 là Cách 2 viết bằng stack (đẩy con đi sau trước để con đi trước nằm trên đỉnh). Cách 4 xác định lật bằng `pos[left] > pos[right]` rồi **kiểm tra toàn dãy** — `order == voyage` — vì nếu mọi cặp anh em đều "được sắp đúng" mà toàn dãy vẫn lệch thì không có nghiệm. Cách 5 dựa vào việc cây con trái/phải chiếm các đoạn liên tiếp có độ dài bằng kích thước của chúng.

**(f) Kiểm chứng bằng máy.** Với **mọi** cây ≤ 4 nút (nhãn `1..n`) và **mọi** hoán vị `voyage` (≈ `8 300` cặp), cộng một mẫu cây 5 nút, mọi cách khớp oracle "thử mọi tập nút" **và** oracle này khẳng định nghiệm tối thiểu **duy nhất** (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi tập nút | `O(2^n · n)` | `O(n)` | oracle, `n ≲ 12` |
| 2. DFS đệ quy + con trỏ | `O(n)` | `O(h)` | **đáp án chính** |
| 3. DFS lặp (stack) | `O(n)` | `O(n)` | tránh đệ quy |
| 4. Suy ra từ `pos` | `O(n)` | `O(n)` | cần kiểm tra toàn dãy cuối |
| 5. Chia đoạn theo kích thước | `O(n)` | `O(n)` | so một giá trị mỗi nút |

Đo thực tế với `n = 100`:

In [18]:
def random_tree(rng, n, labels=None):
    """Cây ngẫu nhiên n nút, nhãn duy nhất (mặc định hoán vị của 1..n)."""
    if n == 0:
        return None
    vals = list(range(1, n + 1)) if labels is None else list(labels)
    if labels is None:
        rng.shuffle(vals)
    it = iter(vals)

    def make(m):
        if m == 0:
            return None
        node = TreeNode(next(it))
        k = rng.randint(0, m - 1)
        node.left = make(k)
        node.right = make(m - 1 - k)
        return node

    return make(n)


def random_flip_copy(root, rng, p=0.5):
    """Bản sao với mỗi nút được lật ngẫu nhiên; trả (cây mới, tập giá trị nút CÓ HAI CON bị lật)."""
    flipped = set()

    def go(n):
        if n is None:
            return None
        l, r = go(n.left), go(n.right)
        if rng.random() < p:
            if l is not None and r is not None:
                flipped.add(n.val)
            l, r = r, l
        return TreeNode(n.val, l, r)

    return go(root), flipped


def bench(fn, root, voyage, repeat=100):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(root, voyage)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(971)
tr = random_tree(rng, 100)
tr2, fl = random_flip_copy(tr, rng)
vo = preorder(tr2)
for name, fn in [("dfs", flip_dfs), ("iter", flip_iter), ("positions", flip_positions), ("sizes", flip_sizes)]:
    assert sorted(fn(tr, vo)) == sorted(fl)
    print(f"{name:10s}: {bench(fn, tr, vo) * 1e6:8.1f} us (n = 100)")

dfs       :     18.7 us (n = 100)
iter      :     18.2 us (n = 100)
positions :     38.5 us (n = 100)
sizes     :     36.6 us (n = 100)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case (kể cả `[-1]`), (C) phản ví dụ, (D) vét cạn mọi cây ≤ 4 nút × mọi `voyage`, (E) tính chất / ngẫu nhiên (cây tới 100 nút, seed cố định), (F) giới hạn.

Đáp án là **tập** nút lật nên so sánh sau `sorted`; `[-1]` giữ nguyên. Ngoài so sánh với oracle, hàm kiểm tra tính hợp lệ `valid_answer` áp dụng các phép lật lên bản sao của cây và so `preorder` với `voyage`.

### A. Ví dụ của đề

In [19]:
def norm(res):
    assert len(res) == len(set(res)), "kết quả có giá trị trùng"
    return sorted(res)


def valid_answer(root, voyage, res):
    """res hợp lệ: [-1], hoặc tập nút thật sự có hai con và việc lật khiến preorder khớp voyage."""
    if res == [-1]:
        return True
    tree_vals = {}
    stack = [root]
    while stack:
        n = stack.pop()
        if n:
            tree_vals[n.val] = n
            stack.extend([n.left, n.right])
    if not all(v in tree_vals for v in res):
        return False
    return preorder(apply_flips(root, res)) == list(voyage)


IMPLS = {
    "dfs": flip_dfs,
    "iter": flip_iter,
    "positions": flip_positions,
    "sizes": flip_sizes,
    "brute": flip_brute,
    "Solution": lambda r, v: Solution().flipMatchVoyage(r, list(v)),
    "SolutionDFS": lambda r, v: SolutionDFS().flipMatchVoyage(r, list(v)),
    "SolutionIter": lambda r, v: SolutionIter().flipMatchVoyage(r, list(v)),
    "SolutionPositions": lambda r, v: SolutionPositions().flipMatchVoyage(r, list(v)),
    "SolutionSizes": lambda r, v: SolutionSizes().flipMatchVoyage(r, list(v)),
    "SolutionBrute": lambda r, v: SolutionBrute().flipMatchVoyage(r, list(v)),
}
SLOW = {"brute", "SolutionBrute"}


def check(l, voyage, want=None, names=None):
    root = build_tree(l)
    n = size(root)
    for name in (names or IMPLS):
        if name in SLOW and n > 10:
            continue
        got = IMPLS[name](build_tree(l), list(voyage))
        got = got if got == [-1] else norm(got)
        if want is not None:
            assert got == want, (name, l, voyage, got, want)
        assert valid_answer(root, voyage, got), (name, l, voyage, got)
    return want


for (l, v), w in ex:
    check(l, v, w)
print("ví dụ đề: brute và", len(IMPLS), "cài đặt đúng; kết quả qua hàm kiểm tra hợp lệ")

ví dụ đề: brute và 11 cài đặt đúng; kết quả qua hàm kiểm tra hợp lệ


### B. Edge case

- `n = 1`: `[1]` với `voyage = [1]` ⇒ `[]`.
- Cây chuỗi (mỗi nút một con): preorder không đổi dù lật ⇒ `[]` (**không** lật nhầm).
- Gốc không khớp `voyage[0]` ⇒ `[-1]`.
- Cần lật nhiều nút (mọi nút hai con).
- Lật được ở trên nhưng thất bại ở dưới ⇒ `[-1]` (không trả danh sách dở dang).
- Nút chỉ có con phải.
- `voyage` là hoán vị bất khả thi.

In [20]:
edge = [
    ([1], [1], []),
    ([1, 2], [1, 2], []),
    ([1, None, 2], [1, 2], []),
    ([1, 2, None, 3], [1, 2, 3], []),
    ([1, None, 2, None, 3], [1, 2, 3], []),
    ([1, 2], [2, 1], [-1]),
    ([1, 2, 3], [1, 3, 2], [1]),
    ([1, 2, 3], [1, 2, 3], []),
    ([1, 2, 3], [1, 3, 4], [-1]),                    # lật 1 rồi thất bại ở nút 2
    ([1, 2, 3], [2, 1, 3], [-1]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 3, 6, 7, 2, 5, 4], [1, 2]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 3, 7, 6, 2, 5, 4], [1, 2, 3]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 2, 5, 4, 3, 7, 6], [2, 3]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 2, 4, 5, 3, 7, 6], [3]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6, 7], [-1]),
    ([1, 2, 3, 4], [1, 2, 4, 3], []),                # 1(2(4,.),3) có preorder [1,2,4,3]
    ([1, 2, 3, 4], [1, 3, 2, 4], [1]),               # lật gốc: 1(3, 2(4))
    ([1, 2, 3, 4], [1, 2, 3, 4], [-1]),              # 4 là con của 2 nên phải đứng ngay sau 2
]
for l, v, w in edge:
    root = build_tree(l)
    if size(root) <= 7:
        assert flip_brute(build_tree(l), v) == w, (l, v)         # kỳ vọng viết tay khớp oracle định nghĩa
    check(l, v, w)
print(len(edge), "edge case (kỳ vọng viết tay đối chiếu oracle brute): mọi cài đặt đúng")

18 edge case (kỳ vọng viết tay đối chiếu oracle brute): mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [21]:
def bug_no_value_check(root, voyage):                # bỏ kiểm tra node.val != voyage[idx] ở nút hiện tại
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    dfs(root)
    return flips


def bug_partial_list(root, voyage):                  # thất bại nhưng vẫn trả danh sách lật dở dang
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    dfs(root)
    return flips


def bug_empty_for_impossible(root, voyage):          # thất bại thì trả [] thay vì [-1]
    res = flip_dfs(root, voyage)
    return [] if res == [-1] else res


def bug_flip_single_child(root, voyage):             # lật cả nút chỉ có con phải
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        nxt = voyage[idx] if idx < len(voyage) else None
        if (node.left is None and node.right) or (node.left and node.left.val != nxt):
            flips.append(node.val)
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    return flips if dfs(root) else [-1]


def bug_wrong_value_recorded(root, voyage):          # ghi giá trị con trái thay vì nút cha
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.left.val)
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    return flips if dfs(root) else [-1]


def bug_no_reorder(root, voyage):                    # quyết định lật nhưng vẫn đi trái trước
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)
        return dfs(node.left) and dfs(node.right)

    return flips if dfs(root) else [-1]


def bug_local_index(root, voyage):                   # con trỏ cục bộ, không dùng chung giữa các lời gọi
    flips = []

    def dfs(node, idx):
        if node is None:
            return idx
        if idx is False or idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        first, second = node.left, node.right
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)
            first, second = node.right, node.left
        a = dfs(first, idx)
        return False if a is False else dfs(second, idx)       # BUG: dùng lại idx cũ cho nhánh thứ hai

    return flips if dfs(root, 0) is not False else [-1]


BUGS = [bug_no_value_check, bug_partial_list, bug_empty_for_impossible, bug_flip_single_child, bug_wrong_value_recorded, bug_no_reorder, bug_local_index]
probe = [([1, 2, 3], [1, 3, 2]), ([1, 2], [2, 1]), ([1, 2, 3], [1, 3, 4]), ([1, None, 2], [1, 2]), ([1, 2, 3, 4, 5, 6, 7], [1, 3, 6, 7, 2, 5, 4]),
         ([1, 2, 3], [1, 2, 3]), ([1, 2, 3, 4], [1, 2, 4, 3]), ([1, 2, 3, 4], [1, 3, 2, 4])]
for bug in BUGS:
    found = None
    for l, v in probe:
        want = flip_dfs(build_tree(l), v)
        try:
            got = bug(build_tree(l), list(v))
            got = got if got == [-1] else sorted(got)
        except Exception as e:
            got = type(e).__name__
        if got != (want if want == [-1] else sorted(want)):
            found = (l, v, got, want)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:28s} sai trên root={found[0]}, voyage={found[1]}: ra {found[2]}, đúng {found[3]}")

bug_no_value_check           sai trên root=[1, 2], voyage=[2, 1]: ra [1], đúng [-1]
bug_partial_list             sai trên root=[1, 2], voyage=[2, 1]: ra [], đúng [-1]
bug_empty_for_impossible     sai trên root=[1, 2], voyage=[2, 1]: ra [], đúng [-1]
bug_flip_single_child        sai trên root=[1, None, 2], voyage=[1, 2]: ra [1], đúng []
bug_wrong_value_recorded     sai trên root=[1, 2, 3], voyage=[1, 3, 2]: ra [2], đúng [1]
bug_no_reorder               sai trên root=[1, 2, 3], voyage=[1, 3, 2]: ra [-1], đúng [1]
bug_local_index              sai trên root=[1, 2, 3], voyage=[1, 3, 2]: ra [-1], đúng [1]


### D. Vét cạn mọi cây ≤ 4 nút × mọi `voyage`

`all_trees(labels)` sinh mọi cây dùng **đúng** tập nhãn `labels` (nhãn duy nhất). Với mỗi cây ta dựng bảng `preorder → các tập lật tối thiểu` **theo đúng định nghĩa** (thử `2^n` tập nút, tính preorder, giữ các tập cùng cỡ nhỏ nhất) và khẳng định mỗi preorder đạt được có **đúng một** tập tối thiểu (tính duy nhất). Sau đó mọi hoán vị `voyage` của `1..n`: nếu có trong bảng ⇒ nghiệm là tập đó; không có ⇒ `[-1]`. Số cặp (cây, `voyage`): `n = 1..4`.

In [22]:
def all_trees(labels):
    labels = tuple(labels)
    if not labels:
        return [None]
    out = []
    for i, root in enumerate(labels):
        rest = labels[:i] + labels[i + 1:]
        for mask in range(1 << len(rest)):
            left = tuple(rest[j] for j in range(len(rest)) if mask >> j & 1)
            right = tuple(rest[j] for j in range(len(rest)) if not mask >> j & 1)
            for lt in all_trees(left):
                for rt in all_trees(right):
                    out.append((root, lt, rt))
    return out


def tup_flip(t, flipset):
    if t is None:
        return None
    v, l, r = t
    l2, r2 = tup_flip(l, flipset), tup_flip(r, flipset)
    return (v, r2, l2) if v in flipset else (v, l2, r2)


def tup_pre(t, out=None):
    out = [] if out is None else out
    if t is not None:
        out.append(t[0])
        tup_pre(t[1], out)
        tup_pre(t[2], out)
    return out


def min_table(t, n):
    """preorder (tuple) -> danh sách các tập lật tối thiểu, tính theo định nghĩa trên mọi 2^n tập nút."""
    best = {}
    for k in range(n + 1):
        for c in itertools.combinations(range(1, n + 1), k):
            p = tuple(tup_pre(tup_flip(t, set(c))))
            if p not in best:
                best[p] = (k, [c])
            elif best[p][0] == k:
                best[p][1].append(c)
    return best


t0 = time.perf_counter()
fast = [n for n in IMPLS if n not in SLOW]
pairs = 0
nonneg = 0
for n in range(1, 5):
    for t in all_trees(range(1, n + 1)):
        table = min_table(t, n)
        for p, (k, sets) in table.items():
            assert len(sets) == 1, (t, p, sets)                       # nghiệm tối thiểu duy nhất
        for voyage in itertools.permutations(range(1, n + 1)):
            want = list(table[voyage][1][0]) if voyage in table else [-1]
            for name in fast:
                got = IMPLS[name](from_tuple(t), list(voyage))
                got = got if got == [-1] else sorted(got)
                assert got == want, (name, t, voyage, got, want)
            if n <= 3:
                for name in SLOW:
                    got = IMPLS[name](from_tuple(t), list(voyage))
                    got = got if got == [-1] else sorted(got)
                    assert got == want, (name, t, voyage, got, want)
            pairs += 1
            nonneg += want != [-1]
print(pairs, "cặp (cây ≤ 4 nút, voyage):", len(fast), "cài đặt nhanh khớp oracle định nghĩa; số cặp có nghiệm:", nonneg, ";", round(time.perf_counter() - t0, 1), "s")

8253 cặp (cây ≤ 4 nút, voyage): 9 cài đặt nhanh khớp oracle định nghĩa; số cặp có nghiệm: 521 ; 0.3 s


In [23]:
# 5 nút: mẫu 300 cây (trong 5040 cây nhãn 1..5) × mọi 120 hoán vị voyage
rng = random.Random(97101)
trees5 = all_trees(range(1, 6))
assert len(trees5) == len(set(trees5)) == 42 * 120
t0 = time.perf_counter()
cnt5 = good5 = 0
for t in rng.sample(trees5, 300):
    table = min_table(t, 5)
    for p, (k, sets) in table.items():
        assert len(sets) == 1
    for voyage in itertools.permutations(range(1, 6)):
        want = list(table[voyage][1][0]) if voyage in table else [-1]
        for name in fast:
            got = IMPLS[name](from_tuple(t), list(voyage))
            got = got if got == [-1] else sorted(got)
            assert got == want, (name, t, voyage, got, want)
        cnt5 += 1
        good5 += want != [-1]
print("n = 5:", cnt5, "cặp mẫu: khớp oracle; có nghiệm:", good5, ";", round(time.perf_counter() - t0, 1), "s")

n = 5: 36000 cặp mẫu: khớp oracle; có nghiệm: 527 ; 1.1 s


### E. Tính chất / ngẫu nhiên (cây tới 100 nút)

Seed cố định. Với cây ngẫu nhiên nhãn `1..n`:

1. **Xây dựng thuận:** lật ngẫu nhiên một số nút của bản sao, lấy `voyage = preorder(bản sao)` ⇒ đáp án phải là **đúng tập các nút có hai con đã bị lật**.
2. **Hàm kiểm tra hợp lệ** `valid_answer` đúng cho mọi kết quả.
3. `voyage` là hoán vị ngẫu nhiên ⇒ nếu kết quả khác `[-1]` thì áp dụng lật phải cho `preorder = voyage`; với `n ≤ 8` so oracle brute.
4. Đổi chỗ hai phần tử của `voyage` hợp lệ: bốn cách khớp nhau; với `n ≤ 9` khớp oracle brute.
5. `voyage` hợp lệ: đáp án **rỗng** ⇔ `voyage == preorder(root)`.

In [24]:
rng = random.Random(20240971)
t0 = time.perf_counter()
cnt_neg = cnt_pos = 0
for it in range(1200):
    n = rng.randint(1, 100) if it % 4 else rng.randint(1, 9)
    tree = random_tree(rng, n)
    cp, expected = random_flip_copy(tree, rng, rng.choice([0.0, 0.3, 0.5, 1.0]))
    voyage = preorder(cp)
    for name in fast:
        got = IMPLS[name](tree, list(voyage))
        assert got != [-1] and norm(got) == sorted(expected), (name, it)                    # (1)
        assert valid_answer(tree, voyage, got)                                              # (2)
    assert (Solution().flipMatchVoyage(tree, list(voyage)) == []) == (voyage == preorder(tree))     # (5)
    # (3) hoán vị ngẫu nhiên
    perm = list(range(1, n + 1))
    rng.shuffle(perm)
    results = {name: IMPLS[name](tree, list(perm)) for name in fast}
    results = {k: (v if v == [-1] else sorted(v)) for k, v in results.items()}
    assert len({tuple(v) for v in results.values()}) == 1, results
    res = results["dfs"]
    assert valid_answer(tree, perm, res)
    cnt_neg += res == [-1]
    if n <= 8:
        assert res == flip_brute(tree, perm)
    # (4) đổi chỗ hai phần tử
    if n >= 2:
        w = voyage[:]
        i, j = rng.sample(range(n), 2)
        w[i], w[j] = w[j], w[i]
        results = {name: IMPLS[name](tree, list(w)) for name in fast}
        results = {k: (v if v == [-1] else sorted(v)) for k, v in results.items()}
        assert len({tuple(v) for v in results.values()}) == 1, results
        assert valid_answer(tree, w, results["dfs"])
        if n <= 9:
            assert results["dfs"] == flip_brute(tree, w)
        cnt_pos += results["dfs"] != [-1]
print("1200 cây ngẫu nhiên (n ≤ 100): 5 tính chất đúng; hoán vị ngẫu nhiên cho [-1]:", cnt_neg, "lần; hoán đổi hai phần tử vẫn có nghiệm:", cnt_pos, "lần;", round(time.perf_counter() - t0, 1), "s")

1200 cây ngẫu nhiên (n ≤ 100): 5 tính chất đúng; hoán vị ngẫu nhiên cho [-1]: 1124 lần; hoán đổi hai phần tử vẫn có nghiệm: 10 lần; 1.2 s


### F. Giới hạn

`n = 100`: chuỗi trái/phải, cây đầy đủ lật **mọi** nút có hai con, cây lệch (comb), và các `voyage` bất khả thi ở **cuối** cây (để kiểm tra không trả danh sách dở dang).

In [25]:
def chain(values, side):
    root = None
    for v in reversed(values):
        root = TreeNode(v, root, None) if side == "L" else TreeNode(v, None, root)
    return root


def full_flip_all(root):
    """Lật mọi nút; trả (cây, tập nút có hai con)."""
    two = set()

    def go(n):
        if n is None:
            return None
        l, r = go(n.left), go(n.right)
        if l is not None and r is not None:
            two.add(n.val)
        return TreeNode(n.val, r, l)

    return go(root), two


vals = list(range(1, 101))
for side in "LR":
    c = chain(vals, side)
    for name in fast:
        assert IMPLS[name](c, preorder(c)) == []
        assert IMPLS[name](c, preorder(chain(vals, "R" if side == "L" else "L"))) == []      # chuỗi: lật không đổi preorder
comp = build_tree(list(range(1, 101)))
flipped_tree, two = full_flip_all(comp)
vo = preorder(flipped_tree)
assert size(comp) == 100
for name in fast:
    assert norm(IMPLS[name](comp, list(vo))) == sorted(two), name
# tìm một phép hoán đổi hai phần tử ở CUỐI voyage làm voyage bất khả thi dù đã lật nhiều nút trước đó
bad = None
for a_, b_ in itertools.combinations(range(90, 100), 2):
    cand = vo[:]
    cand[a_], cand[b_] = cand[b_], cand[a_]
    if flip_dfs(comp, cand) == [-1]:
        bad = cand
        break
assert bad is not None
for name in fast:
    assert IMPLS[name](comp, list(bad)) == [-1], name
assert bug_partial_list(comp, list(bad)) != [-1]           # phiên bản sai trả danh sách dở dang
print("n = 100: chuỗi, cây đầy đủ lật hết (", len(two), "nút có hai con); voyage hỏng ở cuối cây -> [-1] (không trả danh sách dở dang)")

n = 100: chuỗi, cây đầy đủ lật hết ( 49 nút có hai con); voyage hỏng ở cuối cây -> [-1] (không trả danh sách dở dang)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bỏ kiểm tra giá trị nút so với `voyage[idx]`

In [26]:
print("bug:", bug_no_value_check(build_tree([1, 2]), [2, 1]), "| đúng:", Solution().flipMatchVoyage(build_tree([1, 2]), [2, 1]))
assert Solution().flipMatchVoyage(build_tree([1, 2]), [2, 1]) == [-1] and bug_no_value_check(build_tree([1, 2]), [2, 1]) != [-1]

bug: [1] | đúng: [-1]


**Vì sao sai:** không phát hiện nút không khớp nên không bao giờ trả `[-1]`. **Đúng:** `node.val != voyage[idx] ⇒ thất bại`.

### Lỗi 2: trả `[]` thay vì `[-1]`, hoặc danh sách lật dở dang

In [27]:
t, v = build_tree([1, 2, 3]), [1, 3, 4]
print("dở dang:", bug_partial_list(t, v), "| trả []:", bug_empty_for_impossible(t, v), "| đúng:", Solution().flipMatchVoyage(t, v))
assert bug_partial_list(t, v) == [1] and bug_empty_for_impossible(t, v) == [] and Solution().flipMatchVoyage(t, v) == [-1]

dở dang: [1] | trả []: [] | đúng: [-1]


**Vì sao sai:** khi thất bại phải trả **đúng** `[-1]`, bỏ mọi nút đã ghi (chúng vô nghĩa vì không có cách lật nào thành công).

### Lỗi 3: lật cả nút chỉ có một con

In [28]:
t, v = build_tree([1, None, 2]), [1, 2]
print("bug:", bug_flip_single_child(t, v), "| đúng:", Solution().flipMatchVoyage(t, v))
assert bug_flip_single_child(t, v) == [1] and Solution().flipMatchVoyage(t, v) == []

bug: [1] | đúng: []


**Vì sao sai:** nút chỉ có một con không đổi preorder khi lật ⇒ lật là **thừa**, vi phạm "ít nút nhất". **Đúng:** chỉ lật khi `node.left` tồn tại và khác giá trị mong muốn.

### Lỗi 4: ghi nhầm giá trị

In [29]:
t, v = build_tree([1, 2, 3]), [1, 3, 2]
print("bug:", bug_wrong_value_recorded(t, v), "| đúng:", Solution().flipMatchVoyage(t, v))
assert bug_wrong_value_recorded(t, v) == [2] and Solution().flipMatchVoyage(t, v) == [1]

bug: [2] | đúng: [1]


**Vì sao sai:** đề yêu cầu giá trị của nút **bị lật** (nút cha), không phải của con.

### Lỗi 5: quyết định lật nhưng vẫn đi trái trước

In [30]:
t, v = build_tree([1, 2, 3]), [1, 3, 2]
print("bug:", bug_no_reorder(t, v), "| đúng:", Solution().flipMatchVoyage(t, v))
assert bug_no_reorder(t, v) == [-1] and Solution().flipMatchVoyage(t, v) == [1]

bug: [-1] | đúng: [1]


**Vì sao sai:** lật nghĩa là duyệt **phải rồi trái**; nếu ghi nhận lật mà vẫn duyệt trái trước thì nhánh sau so lệch. **Đúng:** `dfs(right) and dfs(left)`.

### Lỗi 6 (bonus): con trỏ cục bộ thay vì dùng chung

In [31]:
t, v = build_tree([1, 2, 3]), [1, 2, 3]
print("bug:", bug_local_index(t, v), "| đúng:", Solution().flipMatchVoyage(t, v))
assert bug_local_index(t, v) != Solution().flipMatchVoyage(t, v)

bug: [-1] | đúng: []


**Vì sao sai:** sau khi duyệt xong cây con trái, con trỏ phải **tiến** qua toàn bộ cây con đó; truyền `idx` theo giá trị cho nhánh phải làm nó đọc sai vị trí. **Đúng:** một con trỏ dùng chung (`nonlocal`), hoặc trả về chỉ số mới.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "biến đổi cây (đổi chỗ con) để một phép duyệt cho ra dãy cho trước" → duyệt cây cùng **con trỏ** vào dãy, mỗi quyết định bị ép bởi giá trị kế tiếp.

**Mẹo:**

1. Chỉ nút có **hai** con mới đáng lật; lật nút một con/lá là thừa.
2. Con trỏ chia sẻ (`nonlocal`) cho DFS tiền thứ tự; nhớ `idx < len(voyage)` khi đọc `voyage[idx]`.
3. Khi thất bại trả `[-1]` **thay** cho danh sách đã ghi.
4. So sánh tập kết quả sau `sorted` (đáp án nhiều thứ tự).
5. Oracle theo định nghĩa: thử `2^n` tập nút; hàm kiểm tra: áp dụng lật lên bản sao rồi so preorder.
6. Dựng cây kiểu LeetCode: nhớ nút trống không có con trong danh sách.

**Họ bài cùng pattern:** Flip Equivalent Binary Trees (951), Construct Binary Tree from Preorder and Inorder Traversal (105), Verify Preorder Serialization (331), Check If a String Is a Valid Sequence from Root to Leaves Path (1430), Binary Tree Preorder Traversal (144).

## 15. Biến thể

1. **Đếm số preorder khác nhau** có thể đạt được bằng các phép lật: `2^(số nút có hai con)`.
2. **Trả về cây sau khi lật** (hoặc `None` nếu bất khả thi).
3. **Giá trị `voyage` không cùng tập giá trị với cây** (không còn đảm bảo `1..n`): vẫn trả `[-1]`.

Cài đặt và đối chiếu vét cạn:

In [32]:
def count_preorders(root):
    two = 0
    stack = [root]
    while stack:
        n = stack.pop()
        if n:
            two += n.left is not None and n.right is not None
            stack.extend([n.left, n.right])
    return 2 ** two


def flipped_tree_or_none(root, voyage):
    res = flip_dfs(root, voyage)
    return None if res == [-1] else apply_flips(root, res)


for t in itertools.chain(all_trees(range(1, 5)), all_trees(range(1, 4))):
    n = len(node_values(from_tuple(t)))
    reach = {tuple(tup_pre(tup_flip(t, set(c)))) for k in range(n + 1) for c in itertools.combinations(range(1, n + 1), k)}
    assert len(reach) == count_preorders(from_tuple(t)), t
print("số preorder đạt được = 2^(số nút có hai con): đúng trên mọi cây ≤ 4 nút")

root = build_tree([1, 2, 3, 4, 5, 6, 7])
ft = flipped_tree_or_none(root, [1, 3, 6, 7, 2, 5, 4])
print("cây sau khi lật:", to_level(ft), "| preorder:", preorder(ft))
assert preorder(ft) == [1, 3, 6, 7, 2, 5, 4] and flipped_tree_or_none(root, [2, 1, 3, 4, 5, 6, 7]) is None
# giá trị voyage ngoài tập giá trị của cây (không vi phạm mục tiêu): luôn [-1]
for fn in fast:
    assert IMPLS[fn](build_tree([1, 2, 3]), [1, 2, 9]) == [-1], fn
print("voyage chứa giá trị lạ: mọi cách trả [-1]")

số preorder đạt được = 2^(số nút có hai con): đúng trên mọi cây ≤ 4 nút
cây sau khi lật: [1, 3, 2, 6, 7, 5, 4] | preorder: [1, 3, 6, 7, 2, 5, 4]
voyage chứa giá trị lạ: mọi cách trả [-1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi tập nút | theo định nghĩa | `O(2^n·n)` | `O(n)` | oracle `n ≲ 12` |
| 2. DFS đệ quy + con trỏ | giá trị kế tiếp quyết định lật | `O(n)` | `O(h)` | **đáp án chính** |
| 3. DFS lặp | stack, đẩy con đi sau trước | `O(n)` | `O(n)` | tránh đệ quy |
| 4. `pos` + kiểm tra cuối | lật ⇔ `pos[trái] > pos[phải]` | `O(n)` | `O(n)` | dạng khai báo |
| 5. Chia đoạn theo kích thước | đoạn `voyage` của mỗi cây con | `O(n)` | `O(n)` | so một giá trị mỗi nút |

Mọi cách khớp oracle "thử mọi tập nút" trên toàn bộ cây ≤ 4 nút × mọi `voyage`, và nghiệm tối thiểu luôn duy nhất.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [33]:
print(Solution().flipMatchVoyage(build_tree([1, 2]), [2, 1]),
      Solution().flipMatchVoyage(build_tree([1, 2, 3]), [1, 3, 2]),
      Solution().flipMatchVoyage(build_tree([1, 2, 3]), [1, 2, 3]),
      Solution().flipMatchVoyage(build_tree([1, None, 2]), [1, 2]))

[-1] [1] [] []


### Bài 2 (tính tay)

`root = [1,2,3,4,5,6,7]`, `voyage = [1,3,7,6,2,4,5]`. (a) Nút nào phải lật? (b) Có khớp không? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DFS đệ quy:

In [34]:
def flip_todo(root, voyage):
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        pass            # TODO 1: nếu node.val != voyage[idx] thì trả False; rồi idx += 1
        pass            # TODO 2: nếu node.left tồn tại và node.left.val != voyage[idx]: ghi node.val, đi phải rồi trái
        return True     # TODO 3: trường hợp không lật: đi trái rồi phải

    return flips if dfs(root) else [-1]


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(flip_todo(build_tree(l), v) == w for (l, v), w in ex)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **DFS lặp dùng stack**:

In [35]:
class MySolution:
    def flipMatchVoyage(self, root, voyage):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().flipMatchVoyage(build_tree(l), v) == w for (l, v), w in ex)

### Bài 5 (mini challenge): số preorder khác nhau

Viết `distinct_preorders(root)` trả về **số** dãy preorder khác nhau mà cây có thể đạt được qua mọi tập phép lật. Kiểm bằng cách liệt kê thật (`2^n` tập nút) trên cây nhỏ.

In [36]:
def distinct_preorders(root):
    pass            # TODO


# Khi làm xong:
# assert distinct_preorders(build_tree([1, 2, 3, 4, 5, 6, 7])) == 8 and distinct_preorders(build_tree([1, 2, None, 3])) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: gốc phải bằng `voyage[0]`.
- Mức 2: nút một con không bao giờ cần lật.
- Mức 3: `[-1] [1] [] []`.

**Bài 2**

- Mức 1: viết cây: `1(2(4,5), 3(6,7))`.
- Mức 2: sau `1` là `3` ⇒ lật `1`; trong `3`: con trái `6` nhưng `voyage` có `7` ⇒ lật `3`.
- Mức 3: sau xong cây con `3` đến `2`; `2` có con `4,5` và `voyage` có `4,5` ⇒ không lật. Tập lật `{1, 3}`.

**Bài 3**

- Mức 1: `if idx >= len(voyage) or node.val != voyage[idx]: return False`.
- Mức 2: `idx += 1` rồi kiểm tra `node.left and idx < len(voyage) and node.left.val != voyage[idx]`.
- Mức 3: lật: `flips.append(node.val); return dfs(node.right) and dfs(node.left)`; không lật: `return dfs(node.left) and dfs(node.right)`.

**Bài 4**

- Mức 1: `stack = [root]`, `i = 0`.
- Mức 2: lấy `node`, so `voyage[i]`; nếu lật đẩy `left` trước rồi `right`.
- Mức 3: không lật đẩy `right` trước rồi `left`; thất bại trả `[-1]`.

**Bài 5**

- Mức 1: mỗi nút hai con cho hai lựa chọn.
- Mức 2: các nút một con/lá không tạo preorder mới.
- Mức 3: `2 ** (số nút có hai con)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (Solution().flipMatchVoyage(build_tree([1, 2]), [2, 1]), Solution().flipMatchVoyage(build_tree([1, 2, 3]), [1, 3, 2]),
        Solution().flipMatchVoyage(build_tree([1, 2, 3]), [1, 2, 3]), Solution().flipMatchVoyage(build_tree([1, None, 2]), [1, 2]))
print(vals)
assert vals == ([-1], [1], [], [])
print("Bài 1 OK")

([-1], [1], [], [])
Bài 1 OK


### Bài 2

In [38]:
res = Solution().flipMatchVoyage(build_tree([1, 2, 3, 4, 5, 6, 7]), [1, 3, 7, 6, 2, 4, 5])
print(res)
assert sorted(res) == [1, 3]
assert preorder(apply_flips(build_tree([1, 2, 3, 4, 5, 6, 7]), res)) == [1, 3, 7, 6, 2, 4, 5]
assert flip_brute(build_tree([1, 2, 3, 4, 5, 6, 7]), [1, 3, 7, 6, 2, 4, 5]) == [1, 3]
print("Bài 2 OK: lật 1 và 3, khớp")

[1, 3]
Bài 2 OK: lật 1 và 3, khớp


### Bài 3

In [39]:
def flip_answer(root, voyage):
    flips, idx = [], 0

    def dfs(node):
        nonlocal idx
        if node is None:
            return True
        if idx >= len(voyage) or node.val != voyage[idx]:
            return False
        idx += 1
        if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
            flips.append(node.val)
            return dfs(node.right) and dfs(node.left)
        return dfs(node.left) and dfs(node.right)

    return flips if dfs(root) else [-1]


assert all(flip_answer(build_tree(l), v) == w for (l, v), w in ex)
for n in range(1, 5):
    for t in all_trees(range(1, n + 1)):
        for voyage in itertools.permutations(range(1, n + 1)):
            assert sorted(flip_answer(from_tuple(t), list(voyage))) == sorted(flip_dfs(from_tuple(t), list(voyage)))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def flipMatchVoyage(self, root, voyage):
        flips, i, stack = [], 0, [root]
        while stack:
            node = stack.pop()
            if node is None:
                continue
            if i >= len(voyage) or node.val != voyage[i]:
                return [-1]
            i += 1
            if node.left and i < len(voyage) and node.left.val != voyage[i]:
                flips.append(node.val)
                stack.append(node.left)
                stack.append(node.right)
            else:
                stack.append(node.right)
                stack.append(node.left)
        return flips


assert all(MySolutionAnswer().flipMatchVoyage(build_tree(l), v) == w for (l, v), w in ex)
rng = random.Random(4)
for _ in range(200):
    tree = random_tree(rng, rng.randint(1, 60))
    cp, expected = random_flip_copy(tree, rng)
    assert sorted(MySolutionAnswer().flipMatchVoyage(tree, preorder(cp))) == sorted(expected)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def distinct_preorders_answer(root):
    two, stack = 0, [root]
    while stack:
        n = stack.pop()
        if n:
            if n.left is not None and n.right is not None:
                two += 1
            stack.extend([n.left, n.right])
    return 2 ** two


assert distinct_preorders_answer(build_tree([1, 2, 3, 4, 5, 6, 7])) == 8 and distinct_preorders_answer(build_tree([1, 2, None, 3])) == 1
for t in all_trees(range(1, 5)):
    n = 4
    reach = {tuple(tup_pre(tup_flip(t, set(c)))) for k in range(n + 1) for c in itertools.combinations(range(1, n + 1), k)}
    assert distinct_preorders_answer(from_tuple(t)) == len(reach)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (duyệt cây theo dãy cho trước, mỗi bước bị ép → DFS có con trỏ chung):**

```python
class Solution:
    def flipMatchVoyage(self, root: Optional[TreeNode], voyage: List[int]) -> List[int]:
        flips, idx = [], 0

        def dfs(node):
            nonlocal idx
            if node is None:
                return True
            if idx >= len(voyage) or node.val != voyage[idx]:
                return False
            idx += 1
            if node.left and idx < len(voyage) and node.left.val != voyage[idx]:
                flips.append(node.val)
                return dfs(node.right) and dfs(node.left)
            return dfs(node.left) and dfs(node.right)

        return flips if dfs(root) else [-1]
```

**Checklist trước khi submit:**

- [ ] Kiểm tra `node.val == voyage[idx]` trước khi đi tiếp?
- [ ] Chỉ lật khi `node.left` tồn tại và khác `voyage[idx]` (không lật nút một con)?
- [ ] Khi lật: đi **phải rồi trái**?
- [ ] Con trỏ `idx` **dùng chung** (`nonlocal`)?
- [ ] Thất bại → trả đúng `[-1]` (không phải `[]`, không phải danh sách dở dang)?
- [ ] Đã thử `[1,2]/[2,1]`, `[1,2,3]/[1,3,2]`, `[1,2,3]/[1,2,3]`, nút một con?
- [ ] Đúng tên `class Solution`, `flipMatchVoyage(self, root, voyage)`, không in debug.